# VetChain | Limpieza y auditoría de datos de Mederi Pharma
**Google Colab · Notebook reproducible · Versión 4 — ventas y obsequios.0 — interpretación bajo cada tabla**

Este cuaderno procesa los CSV reales sin encabezados y produce un panel **producto × mes** para experimentos de pronóstico. No entrena aún XGBoost ni afirma resultados de precisión.

**Definición operacional:** pedidos `T` (en tránsito) y `A` (aprobados/en almacén) = *demanda comercial solicitada*. `C` (cancelados), `D` (devueltos a Mederi) y estados vacíos se excluyen del objetivo principal. **Los pedidos no equivalen a ventas facturadas.** La interpretación de `c9` como cantidad despachada y de Kardex `c5` como stock requiere validación.

**Importante:** las columnas se documentaron mediante análisis de archivos sin cabeceras; las unidades de medida deben validarse con la empresa. Los CSV contienen datos comerciales sensibles: no publiques los originales en un repositorio ni compartas el notebook con sus resultados privados.

**Cómo leer el cuaderno:** ejecuta cada celda en orden. Tras cada tabla o gráfico aparece una explicación generada con los resultados de la ejecución: qué significa, qué concluir y qué no afirmar. Los ceros construidos son *ausencia de pedidos registrados*, no prueba de demanda nula. No se entrena ningún modelo en este cuaderno.


**Actualización V4:** Mederi vende principalmente al por mayor y registra obsequios con precio **S/ 0.00**. Se etiquetan por separado las líneas de precio cero. No se presume que todos esos registros se deban a caducidad; no se conocen aún los lotes ni las fechas de vencimiento. La referencia de importe cero requiere validar que `c11` sea precio unitario.


## 1. Preparación del entorno y carga
En Colab ejecuta la celda siguiente, pulsa **Elegir archivos** y sube los CSV `mederi-1-...` hasta `mederi-7-...` (basta uno por tipo). También se admite un ZIP con los siete CSV. No subas las dos copias del detalle: se utilizará solo una de forma explícita.

In [ ]:
import pandas as pd, numpy as np, json, io, zipfile, shutil, re
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display

INPUT = Path('/content/vetchain_input') if Path('/content').exists() else Path('/mnt/data')
OUTPUT = Path('/content/vetchain_resultados') if Path('/content').exists() else Path('/mnt/data/vetchain_resultados')
INPUT.mkdir(exist_ok=True, parents=True); OUTPUT.mkdir(exist_ok=True, parents=True)

try:
    from google.colab import files
    archivos = files.upload()
    for nombre, contenido in archivos.items():
        if nombre.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(contenido)) as z:
                for info in z.infolist():
                    base=Path(info.filename).name
                    if base.lower().endswith('.csv') and re.match(r'mederi-[1-7]-',base,re.I):
                        (INPUT/base).write_bytes(z.read(info))
        elif nombre.lower().endswith('.csv'):
            (INPUT/Path(nombre).name).write_bytes(contenido)
except ImportError:
    print('Ejecución local: coloca los CSV en', INPUT)

print('Archivos encontrados:')
for f in sorted(INPUT.glob('mederi-*-*.csv')): print(' •',f.name)

## 2. Selección de archivos y revisión de esquema
No interpretamos automáticamente los números de columna como encabezados oficiales. Los mapeos usados a continuación corresponden al diccionario inicial de VetChain y deben mantenerse versionados.

In [ ]:
def localizar(indice):
    candidatos=sorted(INPUT.glob(f'mederi-{indice}-*.csv'))
    if not candidatos:
        raise FileNotFoundError(f'No se encontró mederi-{indice}-*.csv en {INPUT}')
    if len(candidatos)>1:
        print(f'ADVERTENCIA: {len(candidatos)} archivos para mederi-{indice}; elijo {candidatos[0].name}')
    return candidatos[0]

def leer(indice):
    f=localizar(indice)
    return pd.read_csv(f, header=None, dtype='string', encoding='utf-8-sig', keep_default_na=False)

clientes=leer(1); pedidos=leer(2); detalle=leer(3); productos=leer(4)
kardex=leer(5); cobros=leer(6); vendedores=leer(7)
fuentes={'clientes':clientes,'pedidos':pedidos,'detalle':detalle,'productos':productos,'kardex':kardex,'cobros':cobros,'vendedores':vendedores}
resumen=pd.DataFrame([{'fuente':k,'filas':len(v),'columnas':v.shape[1]} for k,v in fuentes.items()])
display(resumen)
display(pedidos.iloc[:3,[0,1,11]].rename(columns={0:'pedido',1:'fecha',11:'estado'}))
# Esta visualización no muestra RUC, nombre, dirección ni teléfono.

In [ ]:
from IPython.display import Markdown
r = resumen.set_index('fuente')
display(Markdown(f"""### Interpretación de la tabla de fuentes
- **Qué muestra:** los archivos realmente cargados y su número de filas y columnas. Se usan **{len(fuentes)} fuentes**, incluyendo **{len(pedidos):,} pedidos** y **{len(detalle):,} líneas**.
- **Cómo leerla:** una fila de pedidos no equivale a una venta facturada; una fila de detalle representa una línea de producto asociada a un pedido.
- **Hallazgo:** los CSV no tienen encabezados oficiales; los nombres asignados a columnas son hipótesis documentadas en el diccionario.
- **Para el jurado:** esta tabla verifica la trazabilidad del origen y el tamaño de los datos antes de modificarlos.

**Interpretación de la muestra de pedidos:** `pedido`, `fecha` y `estado` son ejemplos del formato original. La muestra confirma apariencia del campo, **no** que todos los registros estén correctos. No mostramos datos personales identificables.
"""))

## 3. Auditoría antes de modificar datos
Revisamos estados, relaciones y claves repetidas. **No se elimina ningún duplicado por simple coincidencia**: una repetición de pedido+número de línea puede ser un error o un registro legítimo.

In [ ]:
pedidos=pedidos.rename(columns={0:'pedido_id',1:'fecha',2:'cliente_id',11:'estado'})
detalle=detalle.rename(columns={0:'pedido_id',4:'linea_id',5:'producto_id',7:'unidad_codigo',8:'cantidad_solicitada',9:'cantidad_c9',11:'precio_unitario',13:'total_linea'})
productos=productos.rename(columns={0:'producto_id',1:'producto_nombre',5:'unidad_catalogo',10:'activo'})

estados=pedidos['estado'].replace('',pd.NA).fillna('VACIO').value_counts()
display(estados.rename_axis('estado').reset_index(name='pedidos'))
fig,ax=plt.subplots(figsize=(7,3));estados.plot.bar(ax=ax);ax.set(title='Pedidos por estado',xlabel='Estado',ylabel='Registros');plt.tight_layout();plt.show()

claves_repetidas=detalle.duplicated(['pedido_id','linea_id'],keep=False)
incidencias=detalle.loc[claves_repetidas,['pedido_id','linea_id','producto_id','cantidad_solicitada']].copy()
incidencias.to_csv(OUTPUT/'incidencias_clave_linea.csv',index=False,encoding='utf-8-sig')

control_inicial={
 'pedidos_id_duplicados':int(pedidos['pedido_id'].duplicated().sum()),
 'claves_linea_extra_repetidas':int(detalle.duplicated(['pedido_id','linea_id']).sum()),
 'registros_en_grupos_de_clave_repetida':int(claves_repetidas.sum()),
 'lineas_sin_pedido':int((~detalle['pedido_id'].isin(pedidos['pedido_id'])).sum()),
 'productos_codigo_duplicados':int(productos['producto_id'].duplicated().sum()),
 'lineas_sin_producto_catalogado':int((~detalle['producto_id'].isin(productos['producto_id'])).sum()),
}
display(pd.DataFrame(control_inicial.items(),columns=['control','valor']))

In [ ]:
from IPython.display import Markdown
vacio=int(estados.get('VACIO',0)); nrep=control_inicial['claves_linea_extra_repetidas']; sinped=control_inicial['lineas_sin_pedido']
display(Markdown(f"""### Interpretación: estados, gráfico y controles de calidad
- **Tabla de estados:** cuenta pedidos por código; T (*en tránsito*), A (*aprobado / almacén*), C (*cancelado*) y D (*devuelto a Mederi*) son interpretaciones reportadas por el equipo y deben cotejarse con documentación del ERP. Existen **{vacio} pedidos sin estado**.
- **Gráfico:** permite identificar el peso de cada estado. El predominio de T no significa que todas sus ventas ya fueron facturadas o cobradas.
- **Tabla de controles:** **{nrep} claves extra** repiten la combinación `(pedido_id, linea_id)`; esto **no prueba automáticamente** duplicidad física de registros. **{sinped} líneas sin pedido vinculado** alertan sobre integridad referencial.
- **Decisión técnica:** no borrar automáticamente filas repetidas; conservarlas en `incidencias_clave_linea.csv` y validar su semántica.
- **Para el jurado:** una auditoría responsable cuantifica anomalías, documenta riesgos y evita correcciones silenciosas.
"""))

## 4. Tipos, fechas, cantidades y uniones
Usaremos `cantidad_solicitada` (detalle `c8`). `cantidad_c9` se conserva en un registro auxiliar y **no se considera venta/despacho comprobado**. No se utilizarán montos financieros como variable objetivo de unidades.

**Nueva regla operativa:** las líneas con precio unitario exactamente cero son **candidatas a obsequio**, de acuerdo con lo explicado por Mederi. Las líneas con precio mayor que cero se clasifican como ventas pagadas *provisionales*. No se borran líneas. `c9` queda como campo técnico no confirmado: el contenido interno de una caja no demuestra su significado. El objetivo de venta pagada será un objetivo alternativo y no sustituye silenciosamente al original.


In [ ]:
o=pedidos[['pedido_id','fecha','estado']].copy()
o['fecha']=pd.to_datetime(o['fecha'],errors='coerce',dayfirst=False)
l=detalle[['pedido_id','linea_id','producto_id','unidad_codigo','cantidad_solicitada','cantidad_c9','precio_unitario','total_linea']].copy()
for campo in ['cantidad_solicitada','cantidad_c9','precio_unitario','total_linea']:
    l[campo]=pd.to_numeric(l[campo].str.replace(',','.',regex=False),errors='coerce')

unido=l.merge(o,on='pedido_id',how='left',validate='many_to_one',indicator=True)
validos=unido.loc[unido['estado'].isin(['T','A']) & unido['fecha'].notna() & (unido['cantidad_solicitada']>0) & unido['producto_id'].notna()].copy()
exclusiones={
 'pedidos_fecha_invalida':int(o['fecha'].isna().sum()),
 'lineas_cantidad_no_numerica':int(l['cantidad_solicitada'].isna().sum()),
 'lineas_cantidad_no_positiva':int((l['cantidad_solicitada']<=0).fillna(False).sum()),
 'lineas_sin_union_pedidos':int((unido['_merge']!='both').sum()),
 'lineas_estados_TA':int(unido['estado'].isin(['T','A']).sum()),
 'lineas_excluidas_por_estado':int((~unido['estado'].isin(['T','A'])).sum()),
 'lineas_validas_utilizadas':len(validos),
}
display(pd.DataFrame(exclusiones.items(),columns=['control','valor']))
print('Fechas T/A:',validos['fecha'].min(), 'a',validos['fecha'].max())
print('Códigos de unidad en líneas válidas:')
display(validos['unidad_codigo'].value_counts().rename_axis('unidad_codigo').reset_index(name='lineas'))

In [ ]:
from IPython.display import Markdown
nva=int(len(validos));total=int(len(unido));fe_ini=validos['fecha'].min().date();fe_fin=validos['fecha'].max().date()
display(Markdown(f"""### Interpretación: depuración y unidades de medida
- **Tabla de exclusiones:** cada control indica cuántas filas presentan una condición de calidad. Las causas pueden superponerse: **no se deben sumar todos los controles como si fueran exclusiones independientes**.
- De **{total:,} líneas relacionadas** quedan **{nva:,} líneas** en el conjunto provisional de demanda T/A (fecha válida, cantidad positiva y código de producto presente).
- **Período analizado:** **{fe_ini}** a **{fe_fin}**. La fecha máxima limita hasta cuándo puede considerarse actualizado un pronóstico operativo.
- **Tabla de unidades:** son **códigos**, no conversiones verificadas a unidades físicas. Antes de comparar productos, Mederi debe confirmar equivalencias para cajas, docenas y unidades.
- **Para el jurado:** la variable objetivo es *cantidad solicitada de pedidos T/A*, no demanda no observada ni facturación confirmada.
"""))

## 5. Auditoría de ventas pagadas y obsequios a S/ 0.00

**Contexto confirmado por Mederi:** el canal principal es mayorista B2B; puede entregar unidades gratuitas de productos próximos a caducar y estas operaciones se registran a **precio cero**.

**Criterio técnico provisional:** `c11 = 0` señala **posible obsequio** y `c11 > 0` **posible venta pagada**; `c11` aún debe validarse con el ERP. Los precios faltantes y negativos quedan para revisión, no se fuerzan a una de las dos clases. El contenido de una presentación, por ejemplo 16 comprimidos por caja, **no** es una segunda cantidad que deba multiplicarse automáticamente.

La siguiente tabla presenta líneas y cantidades **en las unidades comerciales originales por SKU**. No se interpretan estas cantidades agregadas como comprimidos ni se atribuye automáticamente a los obsequios una fecha de vencimiento.


In [ ]:
# Mantener el dato de origen y la trazabilidad de todas las filas
validos['tipo_operacion_provisional'] = np.select(
    [validos['precio_unitario'].eq(0), validos['precio_unitario'].gt(0), validos['precio_unitario'].lt(0)],
    ['posible_obsequio', 'posible_venta_pagada', 'precio_negativo_revisar'],
    default='precio_faltante_revisar'
)
validos['cantidad_posible_venta_pagada'] = np.where(
    validos['tipo_operacion_provisional'].eq('posible_venta_pagada'), validos['cantidad_solicitada'], 0.0
)
validos['cantidad_posible_obsequio'] = np.where(
    validos['tipo_operacion_provisional'].eq('posible_obsequio'), validos['cantidad_solicitada'], 0.0
)
validos['cantidad_pendiente_clasificar'] = np.where(
    validos['tipo_operacion_provisional'].str.endswith('_revisar'), validos['cantidad_solicitada'], 0.0
)

resumen_operaciones = (validos.groupby('tipo_operacion_provisional',dropna=False)
    .agg(lineas=('pedido_id','size'), pedidos=('pedido_id','nunique'),
         cantidad_comercial=('cantidad_solicitada','sum'),
         productos=('producto_id','nunique'))
    .reset_index().sort_values('lineas',ascending=False))
display(resumen_operaciones)
validos[['pedido_id','linea_id','producto_id','fecha','estado','cantidad_solicitada',
         'cantidad_c9','precio_unitario','tipo_operacion_provisional',
         'cantidad_posible_venta_pagada','cantidad_posible_obsequio']].to_csv(
         OUTPUT/'lineas_clasificadas_precio_cero.csv', index=False,encoding='utf-8-sig')

# Comprobación de conservación: todas las cantidades se asignan una sola vez
assert np.allclose(
    validos['cantidad_solicitada'].sum(),
    (validos['cantidad_posible_venta_pagada']+validos['cantidad_posible_obsequio']
    +validos['cantidad_pendiente_clasificar']).sum()
), 'Error en el reparto de cantidades'

from IPython.display import Markdown
num_obs=int((validos['tipo_operacion_provisional']=='posible_obsequio').sum())
num_rev=int(validos['tipo_operacion_provisional'].str.endswith('_revisar').sum())
display(Markdown(f"""### Interpretación de la tabla de operaciones
- **{num_obs:,} líneas** tienen precio exactamente cero y se identifican como **posibles obsequios**.
- **{num_rev:,} líneas** requieren revisar precio faltante o negativo.
- Las cantidades se conservan; no se eliminan los obsequios ni se equiparan unidades sueltas a cajas.
- Esta tabla **no permite identificar cuáles obsequios fueron por caducidad** porque aún falta información de lotes/vencimientos.
- Si `c11` no fuera el precio real de la línea, esta clasificación debe corregirse antes de presentar el modelo.
"""))


### Tabla complementaria: obsequios por producto

Se muestran los códigos con mayor cantidad registrada a precio cero, **sin comparar su contenido interno**. Este listado facilita investigar si las salidas promocionales se concentran en ciertos productos, pero no demuestra que tengan próxima fecha de caducidad.


In [ ]:
por_sku = (validos.groupby('producto_id')
    .agg(cantidad_pagada=('cantidad_posible_venta_pagada','sum'),
         cantidad_obsequio=('cantidad_posible_obsequio','sum'),
         cantidad_revisar=('cantidad_pendiente_clasificar','sum'))
    .reset_index())
por_sku['cantidad_total_registrada']=(por_sku['cantidad_pagada']
                                        +por_sku['cantidad_obsequio']
                                        +por_sku['cantidad_revisar'])
display(por_sku.sort_values('cantidad_obsequio',ascending=False).head(15))

display(Markdown("""### Interpretación de la tabla por producto
- **Cantidad pagada:** cantidad comercial registrada en líneas con precio mayor que cero.
- **Cantidad obsequio:** cantidad comercial registrada en líneas con precio cero.
- **Cantidad por revisar:** líneas con precio faltante o negativo.
- **Total:** permite comprobar que no desaparecieron unidades registradas por el sistema.
- La suma entre diferentes SKU es un indicador operativo de volumen registrado, **no una cantidad física homogénea** si difieren las presentaciones.
"""))


## 5. Construcción del panel mensual
Cada fila corresponde a un producto en un mes. **Los meses sin pedidos se completan con cero solo a partir del primer mes observado para cada producto**, lo cual representa ausencia de pedido registrado, no prueba demanda real igual a cero. Debe vigilarse el desabastecimiento y la discontinuación.

In [ ]:
validos['mes']=validos['fecha'].dt.to_period('M').dt.to_timestamp()
resumen_mes=(validos.groupby(['producto_id','mes'],as_index=False)
             .agg(demanda_solicitada=('cantidad_solicitada','sum'),
                  n_pedidos=('pedido_id','nunique'),
                  n_lineas=('pedido_id','size'),
                  cantidad_c9=('cantidad_c9','sum'),
                  demanda_posible_pagada=('cantidad_posible_venta_pagada','sum'),
                  salida_posible_obsequio=('cantidad_posible_obsequio','sum'),
                  cantidad_por_revisar=('cantidad_pendiente_clasificar','sum')))
ultimo_mes=validos['mes'].max(); partes=[]
for sku,sub in resumen_mes.groupby('producto_id'):
    meses=pd.date_range(sub['mes'].min(),ultimo_mes,freq='MS')
    parte=pd.DataFrame({'producto_id':sku,'mes':meses}).merge(sub,on=['producto_id','mes'],how='left')
    for campo in ['demanda_solicitada','n_pedidos','n_lineas','cantidad_c9','demanda_posible_pagada','salida_posible_obsequio','cantidad_por_revisar']:
        parte[campo]=parte[campo].fillna(0)
    parte['sin_pedidos_en_mes']=(parte['n_pedidos']==0).astype(int)
    partes.append(parte)
panel=pd.concat(partes,ignore_index=True).sort_values(['producto_id','mes']).reset_index(drop=True)
panel['anio']=panel['mes'].dt.year
panel['mes_num']=panel['mes'].dt.month
print('Filas panel:',len(panel),'SKU:',panel['producto_id'].nunique(),'Mes máximo:',panel['mes'].max().date())
display(panel.head())

In [ ]:
from IPython.display import Markdown
nt=len(panel);n0=int((panel['n_pedidos']==0).sum());n1=nt-n0;pct=n0/nt*100
resumen_ceros=(panel.groupby('producto_id',as_index=False).agg(meses_total=('mes','size'),meses_con_pedidos=('n_pedidos',lambda x:int((x>0).sum())),demanda_total=('demanda_solicitada','sum')))
resumen_ceros['pct_meses_sin_pedido']=((1-resumen_ceros['meses_con_pedidos']/resumen_ceros['meses_total'])*100).round(1)
print('Auditoría de ceros en el panel producto-mes:')
display(pd.DataFrame([{'metrica':'Producto-mes totales','valor':nt},{'metrica':'Producto-mes con pedidos','valor':n1},{'metrica':'Producto-mes sin pedidos registrados','valor':n0},{'metrica':'Porcentaje sin pedidos','valor':round(pct,2)}]))
print('Productos con mayor presencia de meses de pedidos:')
display(resumen_ceros.sort_values(['meses_con_pedidos','demanda_total'],ascending=False).head(12))
display(Markdown(f"""### Interpretación de la tabla producto-mes y de los ceros
- **Qué representa:** cada fila es un producto en un mes desde su **primer mes con pedido registrado** hasta el último mes global observado. Se generan meses intermedios para conservar continuidad temporal.
- **Por qué aparecen tantos ceros:** **{n0:,} de {nt:,} filas ({pct:.1f}%)** no tienen pedidos registrados. Esos meses se completan con `fillna(0)`; **no se inventan ventas positivas**.
- **Importante:** cero pedidos registrados **no equivale siempre a cero demanda real**. También podría existir falta de stock, producto discontinuado o registros faltantes.
- **Tabla de productos:** `meses_con_pedidos` cuenta actividad documentada; `pct_meses_sin_pedido` resume intermitencia dentro de la ventana observada por producto, no en toda la vida comercial.
- **Precaución:** antes de su primer pedido no generamos ceros para evitar suponer que el producto estaba comercializado. Al final de su ciclo, meses sin pedidos podrían corresponder a descontinuación no documentada.
- **Consecuencia en Data Science:** comparar modelos de demanda intermitente y segmentar por continuidad de actividad.
"""))

## 6. Ingeniería de variables predictivas sin fuga temporal
Los `lags` y promedios se construyen con `shift(1)` o `shift(k)`; por tanto, para predecir el mes *t* no se usa la demanda observada en ese mismo mes como predictor. `objetivo_demanda_solicitada` es la etiqueta de aprendizaje, **no un predictor**.

*Nota*: estas variables son apropiadas para predicción a un mes. Para horizonte 2/3 meses se requiere generación recursiva o construcción de targets directos y validación sin utilizar observaciones del futuro.

**V4:** los rezagos para modelado se calcularán sobre las posibles ventas pagadas; las cantidades totales T/A y los posibles obsequios permanecerán en el panel para auditoría.


In [ ]:
# Dos objetivos explícitos: cantidad T/A registrada, y cantidad a precio > 0 (provisional).
panel['objetivo_demanda_pagada_provisional'] = panel['demanda_posible_pagada']
panel['objetivo_salidas_obsequio_provisional'] = panel['salida_posible_obsequio']
# La ingeniería de rezagos se basa en el objetivo de ventas pagadas para no mezclar regalos.
grupo=panel.groupby('producto_id',sort=False)['objetivo_demanda_pagada_provisional']
for k in [1,2,3,6,12]: panel[f'lag_{k}m']=grupo.shift(k)
for k in [3,6,12]:
    panel[f'promedio_{k}m']=panel.groupby('producto_id')['objetivo_demanda_pagada_provisional'].transform(lambda s:s.shift(1).rolling(k,min_periods=k).mean())
panel['std_3m']=panel.groupby('producto_id')['objetivo_demanda_pagada_provisional'].transform(lambda s:s.shift(1).rolling(3,min_periods=3).std())
panel['meses_historicos_disponibles']=panel.groupby('producto_id').cumcount()
panel=panel.rename(columns={'demanda_solicitada':'objetivo_demanda_solicitada'})
full=panel.copy();full['mes']=full['mes'].dt.strftime('%Y-%m')
full.to_csv(OUTPUT/'dataset_producto_mes_completo.csv',index=False,encoding='utf-8-sig')
training=full.loc[full['meses_historicos_disponibles']>=12].drop(columns=['cantidad_c9','sin_pedidos_en_mes','n_lineas','n_pedidos','demanda_posible_pagada','salida_posible_obsequio','cantidad_por_revisar','objetivo_salidas_obsequio_provisional','objetivo_demanda_solicitada']).copy()
training.to_csv(OUTPUT/'dataset_entrenamiento_xgboost.csv',index=False,encoding='utf-8-sig')
productos[['producto_id','producto_nombre','unidad_catalogo','activo']].drop_duplicates('producto_id').to_csv(OUTPUT/'catalogo_para_dashboard.csv',index=False,encoding='utf-8-sig')
print('Filas entrenamiento:',len(training));display(training.head(5))
print('OBJETIVO PARA FUTURO ENTRENAMIENTO:', 'objetivo_demanda_pagada_provisional')


In [ ]:
from IPython.display import Markdown
fns=[c for c in training.columns if c.startswith(('lag_','promedio_','std_'))]
display(pd.DataFrame([{'variable':x,'definicion':('Cantidad solicitada '+x.replace('lag_','hace ').replace('m','')+' meses' if x.startswith('lag_') else ('Media de meses ANTERIORES' if x.startswith('promedio_') else 'Desviación de 3 meses ANTERIORES'))} for x in fns]))
display(Markdown(f"""### Interpretación de las variables predictivas
- **`objetivo_demanda_solicitada`:** total de unidades solicitadas en ese producto y mes; es **la etiqueta `y`**, no una entrada del modelo.
- **`lag_1m`, `lag_3m`, `lag_12m`:** observaciones anteriores del mismo producto; un `lag_12m` puede ayudar a detectar repetición anual, **sin demostrar** estacionalidad.
- **`promedio_3m`, `promedio_6m`, `promedio_12m`:** estadísticas exclusivamente de meses previos; sintetizan tendencia de corto y mediano plazo.
- **`std_3m`:** mide variabilidad reciente; elevada dispersión puede indicar dificultad predictiva.
- **`mes_num`, `anio`:** características calendario. Un modelo no debe aprender resultados del futuro por error de preparación.
- **Resultado:** se generan **{len(training):,} filas candidatas para entrenamiento** con 12 meses previos. Una tabla de entrenamiento **no demuestra** que XGBoost esté entrenado ni que WAPE sea bajo.
- **Cautela:** el archivo exportado puede contener campos contextuales como `meses_historicos_disponibles`; al crear `X` separaremos expresamente la variable objetivo y cualquier identificador no pertinente. Durante backtesting deberán evitarse rezagos alimentados con valores reales futuros en predicción multimes.
"""))

## 7. Gráficos exploratorios y corte de datos de 2026

La gráfica histórica presenta demanda **solicitada** en pedidos T/A. El último período observado es enero de 2026, pero **no está confirmado que dicho mes tenga cobertura completa**. Se separa visualmente del tramo anterior y **no debe interpretarse su menor valor como caída de ventas**. Se muestra una segunda gráfica por año con años completos (2019–2025), y un ranking exploratorio de productos.

**Precauciones:** (1) 2026 no es un año comparable con los años completos; (2) los volúmenes entre SKUs pueden estar en distintas unidades de medida; (3) lo registrado no equivale necesariamente a toda la demanda real cuando hay quiebres de stock.


**Actualización V4:** las gráficas utilizan la cantidad de líneas con precio positivo (posibles ventas pagadas), **no** los obsequios. Enero de 2026 permanece marcado como cobertura por validar. Las cantidades entre SKU con presentaciones diferentes no se consideran unidades físicas equivalentes.


In [ ]:
serie_global = panel.groupby('mes')['objetivo_demanda_pagada_provisional'].sum().sort_index()
fecha_corte_grafico = pd.Timestamp('2026-01-01')
historico = serie_global.loc[serie_global.index < fecha_corte_grafico]
recientes = serie_global.loc[serie_global.index >= fecha_corte_grafico]

fig, ax = plt.subplots(figsize=(13,5))
ax.plot(historico.index, historico.values, linewidth=2, marker='.', markersize=3,
        label='Posibles ventas pagadas históricas (hasta dic. 2025)')
if not recientes.empty:
    # No conectar como tramo continuo para evitar atribuir una caída a actividad real.
    ax.scatter(recientes.index, recientes.values, marker='x', s=100,
               label='2026: cobertura por validar', zorder=5)
    for fecha, cantidad in recientes.items():
        ax.annotate('Cobertura por validar', (fecha,cantidad),
                    xytext=(-110,20),textcoords='offset points', fontsize=9)
ax.set(title='Cantidad mensual en ventas a precio positivo (T/A): observaciones recientes sujetas a validación',
       xlabel='Mes', ylabel='Cantidad en unidades originales')
ax.legend()
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

# Comparación anual solo de años completos en el panel. No equipara aún las presentaciones.
serie_2019_2025=serie_global.loc[(serie_global.index.year>=2019)&(serie_global.index.year<=2025)]
anual = serie_2019_2025.groupby(serie_2019_2025.index.year).agg(['sum','size'])
anual.columns=['unidades_solicitadas','meses_observados']
anual['serie_completa_12_meses'] = anual['meses_observados'].eq(12)
display(anual.rename_axis('anio').reset_index())

fig,ax=plt.subplots(figsize=(10,4))
anual['unidades_solicitadas'].plot(kind='bar',ax=ax)
ax.set(title='Suma anual de posibles ventas pagadas T/A: 2019–2025',xlabel='Año',ylabel='Unidades originales')
plt.tight_layout();plt.show()

por_producto = panel.groupby('producto_id')['objetivo_demanda_pagada_provisional'].sum().nlargest(10).sort_values()
fig,ax=plt.subplots(figsize=(9,4))
por_producto.plot.barh(ax=ax)
ax.set(title='10 códigos con mayor cantidad solicitada (unidades originales)',xlabel='Cantidad')
plt.tight_layout();plt.show()
print('Advertencia: comparar cantidades entre SKUs requiere validar unidades y presentaciones.')


In [ ]:
from IPython.display import Markdown
nuevos = len(recientes)
ultimo = serie_global.index.max().strftime('%Y-%m') if len(serie_global) else 'sin registros'
ultimo_valor = float(serie_global.iloc[-1]) if len(serie_global) else float('nan')
display(Markdown(f"""### Interpretación de los gráficos exploratorios
- **Último mes en el panel:** `{ultimo}`, con **{ultimo_valor:,.2f} unidades originales solicitadas** en los pedidos T/A. La cobertura de este período **no se ha verificado**, por lo que no concluimos que haya una caída real de actividad.
- **Gráfica mensual:** separa los **{nuevos} meses de 2026** como observaciones sujetas a validación, no conectadas a la curva histórica. El nivel de enero de 2026 no describe el comportamiento de todo 2026.
- **Resumen anual:** utiliza solo 2019–2025 para comparar sumas anuales. La columna `serie_completa_12_meses` indica si la serie mensual incluye doce posiciones por año; **no demuestra que la captura original estuviera completa**.
- **Ranking por producto:** representa volumen solicitado en unidades originales; requiere verificar si cajas, docenas y unidades son comparables.
- **Para la defensa:** una caída aparente puede reflejar incompletitud de la exportación, cambios de sistema, registro de estados o demanda. Se debe validar el corte de datos antes de formular conclusiones y entrenar para períodos posteriores.
"""))


## 8. Auditoría exportable y documentación de limitaciones
El JSON es **un reporte de auditoría, no el código**. Se excluyen identificadores de clientes de las salidas agregadas. Los originales deben permanecer bajo custodia autorizada.

In [ ]:
auditoria={
 'fuentes':{nombre:len(df) for nombre,df in fuentes.items()},
 'estados':{str(k):int(v) for k,v in estados.items()},
 **control_inicial,**exclusiones,
 'fecha_minima_TA':str(validos['fecha'].min().date()),
 'fecha_maxima_TA':str(validos['fecha'].max().date()),
 'productos_con_demanda':int(resumen_mes['producto_id'].nunique()),
 'filas_producto_mes':len(full),
 'filas_entrenamiento_lag12':len(training),
 'definicion_target':'Cantidad mensual en líneas T/A con precio unitario mayor a cero según c11 (clasificación provisional, sin equivalencias internas por presentación).',
 'lineas_posibles_obsequios':int((validos['tipo_operacion_provisional']=='posible_obsequio').sum()),
 'lineas_precio_para_revision':int(validos['tipo_operacion_provisional'].str.endswith('_revisar').sum()),
 'limites':['Unidades c7 y catálogo por validar','Cantidad c9 pendiente de confirmar','Precio c11 pendiente de confirmación con Mederi','Precio cero no acredita caducidad ni reemplaza los datos de lote','Kardex c5 pendiente de confirmar',
           'Meses sin pedidos imputados a cero desde primer mes','Sin costos o plazos de importación reales',
           'No se ha entrenado ni validado XGBoost en este cuaderno']
}
(OUTPUT/'auditoria_limpieza.json').write_text(json.dumps(auditoria,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([{'salida':f.name,'tamano_kb':round(f.stat().st_size/1024,1)} for f in OUTPUT.glob('*') if f.is_file()]))

In [ ]:
from IPython.display import Markdown
display(Markdown("""### Interpretación de la tabla de archivos generados
- **`dataset_producto_mes_completo.csv`:** panel de análisis que incluye meses completados, variable objetivo y señales descriptivas.
- **`dataset_entrenamiento_xgboost.csv`:** subconjunto con historial previo suficiente y variables rezagadas; requiere validación cronológica antes de entrenar.
- **`catalogo_para_dashboard.csv`:** etiquetas de productos para el frontend; no implica correspondencia validada con laboratorios proveedores.
- **`incidencias_clave_linea.csv`:** casos de clave compuesta repetida; archivo para revisión, no para borrado automático.
- **`auditoria_limpieza.json`:** bitácora estructurada de conteos y advertencias; **JSON no reemplaza el código Python**.

**Advertencia de alcance:** este notebook NO calcula WAPE ni produce predicciones futuras. Su entregable es un dataset auditado de demanda solicitada y una explicación reproducible del tratamiento.
"""))

## 9. Descargar todos los resultados
El ZIP incluye los datasets, auditoría y claves repetidas para revisión. **No incluye los CSV originales** por confidencialidad.

In [ ]:
zip_path=OUTPUT.parent/'VetChain_Resultados_Colab.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for archivo in sorted(OUTPUT.glob('*')):
        if archivo.is_file():z.write(archivo,arcname=archivo.name)
print('Paquete generado:',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Archivo disponible localmente:',zip_path)

## 10. Próximos pasos

1. **Validar `c11`** como precio unitario (se usa provisionalmente para distinguir posibles obsequios de ventas pagadas), y **confirmar `c8`, `c9` y las unidades comerciales**.
2. Solicitar evidencia de lotes y fechas de caducidad para identificar qué bonificaciones corresponden realmente a próximos vencimientos; el precio cero por sí solo no demuestra esa causa.
3. Verificar la cobertura de enero de 2026 y productos descontinuados antes de inferir tendencias.
4. Comparar un modelo XGBoost global con métodos de referencia y enfoques para demanda intermitente, mediante pruebas temporales sin fuga de información.
5. Calcular WAPE por horizonte y segmento. Sin validación, las metas de precisión siguen siendo objetivos hipotéticos.
6. Separar pronóstico comercial de salidas promocionales y añadir inventario, tránsito, vencimientos y plazos **reales** antes de recomendar importaciones.

**No se ha entrenado ningún modelo en este notebook.**
